# 05 · M1.2 — Student-t fitted on a window of 3 months, 6 months, 1 year or 2 years

## Executive summary (read this first)

**M1.2 = M1.1 with a Student-t step distribution fitted on a window**, everything else as M1.1:

| | M0 | M1.1 | M1.2 |
|---|---|---|---|
| step (transform) | change | per asset type (`transformations.toml`) | per asset type |
| centre | last + h × mean step (drift) | same | same (**M0's drift kept**, no drift rule) |
| distribution of a step | normal | normal | **Student-t** on rates, FX, equity factors; normal on CPI, NFP, UNRATE |
| fitted on | last 300 steps | last 300 steps | last **W** steps, W = 3 months, 6 months, 1 year or 2 years |

**What "fitted on W steps" means:** the drift (mean step), the sd and the correlation between assets, and the Student-t's ν all
come from the last W daily steps. ν comes from the window's tail weight: a Student-t's excess kurtosis is `6 ÷ (ν − 4)`, so
`ν = 4 + 6 ÷ k`, with k the window's mean excess kurtosis (ν kept in 4.2–30; a window with k < 0.2 stays normal). The
Student-t multiplies each draw's normal shock by `√((ν − 2) ÷ W_χ)`, `W_χ ~ χ²(ν)`, one per draw shared by the unit's
Student-t cells: the variance is unchanged and the tails get fatter.

| window | daily steps |
|---|---|
| 3 months | 63 |
| 6 months | 126 |
| 1 year | 252 |
| (M0's window) | 300 |
| 2 years | 504 |

The four monthly units (CPI, NFP, UNRATE) keep M0's 300 rows: 3–24 monthly rows are too few to fit a distribution.

**The test:** notebook 01's backtest dates and answers; competition score per unit = `Σ weight × mean(category score) ÷
mean(M0's)`. Below 1 beats M0. The windows are compared side by side, not chosen: picking the best of them on this block
would make that number slightly optimistic.

Results: §5 (per family and asset type), §6 (per unit), §8 (in short). No realized outcome is read.

## 0 — Setup: notebook 01's backtest dates, answers, M0 and M1.1 scores; the rules

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
WINDOWS = {63: "3 months", 126: "6 months", 252: "1 year", 300: "300 rows (M0)", 504: "2 years"}

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"]}
    return {"transform": RULES["default"]["log_return"] if target_type == "log_return" else base["transform"], "shock": base["shock"]}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "monthly": uid in m0._MONTHLY_STEPS,
            "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
DATES = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    DATES[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {len(M0_SCORES):,} backtest dates; windows {WINDOWS}")

103 units, 23,635 backtest dates; windows {63: '3 months', 126: '6 months', 252: '1 year', 300: '300 rows (M0)', 504: '2 years'}


## 1 — How fat are the tails, per window?

Excess kurtosis of the daily steps (0 = normal; a Student-t with ν = 5 has 6), and the ν it gives, at each unit's real
as-of, per window: a short window sees fewer extreme days, so it usually reads thinner tails.

In [2]:
def window_steps(U, a, origin, W):
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (W + 1)): k]                 # W + 1 rows -> W steps
    lm = U["rules"][a]["transform"] == "log_diff" and min(x for _, x in rows) > 0
    st, lv = m0._steps([(d, np.log(x)) for d, x in rows] if lm else rows, "level" if lm else U["target_type"])
    return st, lm, rows[-1][1], lv

def fit_nu(x, rows_t):
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < 0.2 else float(np.clip(4 + 6 / k, 4.2, 30))

rows = []
for u, U in UNITS_DATA.items():
    if U["monthly"]:
        continue
    for W in WINDOWS:
        xs = [window_steps(U, a, U["asof"], W)[0] for a in U["assets"]]
        common = sorted(set.intersection(*(set(s_) for s_ in xs)))
        x = np.array([[s_[d] for d in common] for s_ in xs])
        t_rows = [i for i, a in enumerate(U["assets"]) if U["rules"][a]["shock"] == "student_t"]
        z = x - x.mean(axis=1, keepdims=True)
        rows.append({"type": TYPE[u], "window": WINDOWS[W],
                     "excess kurtosis": float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3)),
                     "nu": fit_nu(x, t_rows)})
K = pd.DataFrame(rows)
K["window"] = pd.Categorical(K["window"], list(WINDOWS.values()))
display(K.groupby(["type", "window"], observed=True).agg(kurtosis=("excess kurtosis", "median"), nu=("nu", "median"),
                                                         normal_share=("nu", lambda v: v.isna().mean())).unstack("window"))

kurtosis                                             nu                                       normal_share                                      
window        3 months 6 months 1 year 300 rows (M0) 2 years 3 months 6 months 1 year 300 rows (M0) 2 years     3 months 6 months 1 year 300 rows (M0) 2 years
type                                                                                                                                                          
equity_factor   0.1919   0.2033  1.515         1.923   3.169    11.95    6.979  6.459          6.95   5.898          0.5      0.5 0.1875        0.0625       0
fx               1.007     1.38  1.552         1.536   2.546    8.833    8.312  7.866         7.885   6.356          0.2   0.1143      0       0.02857       0
fx+rates         1.176    1.304  1.975         1.858   2.884     6.92    7.946  7.038          7.23    6.08          0.4      0.2      0             0       0
rates            0.295   0.4963  1.023         1.001   1.546    11.33    11.31  9.096         9.365   7.856       0.4419   0.2791 0.1163       0.09302 0.02326

## 2 — M1.2 at any date

M1.1's code with the window as a setting and the Student-t added. Check: with W = 300 and the Student-t off, M1.2 is M1.1
(and so equals M0 exactly on the units without an FX or CPI asset).

In [3]:
def m12_draws(U, origin, W=300, student_t=True):
    """M1.2's 500 draws at `origin`, shape (500, n_assets, n_horizons). W: steps to fit on (monthly units: 300)."""
    W = m0._WINDOW if U["monthly"] else W
    assets, horizons, tt = U["assets"], U["horizons"], U["target_type"]
    by_asset = sorted(assets); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - W): k] if W == m0._WINDOW else U["rows"][a][max(0, k - (W + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(x for _, x in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(x)) for d, x in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, tt); last[a] = lv if tt == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    mean = np.array([(0.0 if log_mode[a] else last[a]) + s[h] * mu[ai[a]] for a, h in cells])      # M0's drift
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"]) if student_t else None
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

same_as_m0 = [u for u, U in UNITS_DATA.items() if "log_diff" not in [r["transform"] for r in U["rules"].values()]]
d = max(float(np.abs(m12_draws(UNITS_DATA[u], UNITS_DATA[u]["asof"], 300, False) - m0.draws(UNITS / u)).max()) for u in same_as_m0)
print(f"W = 300, Student-t off, on the {len(same_as_m0)} units where M1.1 = M0: max |M1.2 - M0| = {d:.1e}")
assert d == 0

W = 300, Student-t off, on the 61 units where M1.1 = M0: max |M1.2 - M0| = 0.0e+00


## 3 — Score M1.2 at every backtest date, for each window

In [4]:
cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)

def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": o, "window": W, **parts(U["card"], m12_draws(U, o, W).reshape(500, -1), y)}
            for o, y in DATES[u] for W in WINDOWS]

t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    M12 = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored M1.2 on {len(M12) // len(WINDOWS):,} dates x {len(WINDOWS)} windows in {time.time() - t0:.0f}s")

scored M1.2 on 23,635 dates x 5 windows in 7s


## 4 — The competition score per unit

`score = Σ w_k × mean(category_k) ÷ mean(M0's category_k)` over the unit's backtest dates.

In [5]:
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

SC = {"M0": pd.Series(1.0, index=UNIT_IDS), "M1.1": vs_m0(M11_SCORES)}
for W, label in WINDOWS.items():
    SC[f"M1.2, {label}"] = vs_m0(M12[M12.window == W])
PER_UNIT = pd.DataFrame(SC)
PER_UNIT.insert(0, "asset type", pd.Series(TYPE)); PER_UNIT.insert(0, "family", pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS}))

## 5 — M0, M1.1 and M1.2 per family and per asset type

The mean competition score over units (as on the leaderboard), and the number of units each beats M0 on.

In [6]:
models = [c for c in PER_UNIT.columns if c not in ("family", "asset type")]
def table(by):
    t = PER_UNIT.groupby(by)[models].mean()
    t.loc["all"] = PER_UNIT[models].mean()
    return t
FAMILY, BY_TYPE = table("family"), table("asset type")
better = pd.Series({m: f"{(PER_UNIT[m] < 1).sum()}/{len(PER_UNIT)}" for m in models}, name="better than M0")
better_m11 = pd.Series({m: f"{(PER_UNIT[m] < PER_UNIT['M1.1']).sum()}/{len(PER_UNIT)}" for m in models}, name="better than M1.1")
print("per family (rows) x model (columns):"); display(FAMILY.T.assign(**{"better than M0": better, "better than M1.1": better_m11}))
print("per asset type (rows) x model (columns):"); display(BY_TYPE.T)

per family (rows) x model (columns):


family,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0,better than M1.1
M0,1,1,1,1,1,0/103,20/103
M1.1,0.9955,0.999,0.9993,0.9968,0.9976,25/103,0/103
"M1.2, 3 months",1.517,1.205,1.37,1.097,1.277,8/103,6/103
"M1.2, 6 months",1.162,1.116,1.143,1.036,1.108,19/103,19/103
"M1.2, 1 year",1.008,1.017,1.004,0.998,1.006,45/103,40/103
"M1.2, 300 rows (M0)",0.9911,0.9995,0.9982,0.993,0.9954,62/103,60/103
"M1.2, 2 years",0.97,0.9967,0.9761,1.01,0.9902,64/103,60/103


per asset type (rows) x model (columns):


asset type,cpi,equity_factor,fx,fx+rates,payrolls,rates,unemployment,all
M0,1,1,1,1,1,1,1,1
M1.1,0.8727,1,1.001,0.9948,1,1,1,0.9976
"M1.2, 3 months",0.8727,1.206,1.249,1.345,1,1.351,1,1.277
"M1.2, 6 months",0.8727,1.045,1.107,1.151,1,1.144,1,1.108
"M1.2, 1 year",0.8727,1.002,1.015,0.9767,1,1.011,1,1.006
"M1.2, 300 rows (M0)",0.8727,0.9939,0.9994,0.992,1,0.9985,1,0.9954
"M1.2, 2 years",0.8727,0.9909,0.9576,0.9759,1,1.023,1,0.9902


## 6 — Per unit

In [7]:
PER_UNIT.rename_axis("unit").reset_index().set_index(["family", "unit"]).sort_index()

asset type  M0   M1.1  M1.2, 3 months  M1.2, 6 months  M1.2, 1 year  M1.2, 300 rows (M0)  M1.2, 2 years
family unit                                                                                                                                         
T2-F1  t2-F1-ai-mom-2024                  equity_factor   1      1           1.514           1.026          1.02                1.009         0.8672
       t2-F1-aud-on-hold-2016                        fx   1 0.9955           1.706           1.256         1.086               0.9953         0.9286
       t2-F1-cad-boc-2017                            fx   1   1.02           1.894           1.245        0.9728                1.004         0.9361
       t2-F1-chf-highly-valued-2021                  fx   1  1.037           1.892           1.207        0.9874                0.978         0.9507
       t2-F1-conflicting-texts-2024               rates   1      1           1.614           1.226         1.018               0.9973          1.096
       t2-F1-considerable-period-2003             rates   1      1           2.404           1.674        0.9668               0.9851         0.8052
       t2-F1-conundrum-2005                       rates   1      1           1.684          0.9914         1.064               0.9961         0.8843
       t2-F1-cpi-glidepath-2023                     cpi   1 0.8517          0.8517          0.8517        0.8517               0.8517         0.8517
       t2-F1-dkk-peg-2019                            fx   1  1.006           1.238           1.069         1.039                1.003         0.8463
       t2-F1-eur-range-2017                          fx   1 0.9528           1.295           1.059        0.9725               0.9536         0.8289
       t2-F1-fed-put-2019                 equity_factor   1      1            1.49           1.033        0.9891               0.9858          1.038
       t2-F1-greater-confidence-2024              rates   1      1            1.01          0.9708        0.9954                1.004          1.007
       t2-F1-hawkish-cut-2024                     rates   1      1           1.027           0.963        0.9882                1.007          1.042
       t2-F1-jpy-ycc-flex-2018                       fx   1  1.034           1.802           1.441         1.104                1.039         0.8853
       t2-F1-liftoff-telegraph-2015               rates   1      1           1.572            1.33        0.9423               0.9795          1.112
       t2-F1-measured-pace-2004                   rates   1      1          0.9226           1.005         1.022                1.001          1.001
       t2-F1-normalization-2017                   rates   1      1           1.445           1.071         1.042                1.007         0.9681
       t2-F1-patient-dropped-2015                 rates   1      1           1.734           1.396        0.9863               0.9984          1.199
       t2-F1-pause-2006                           rates   1      1           1.845            1.03         1.022               0.9942          1.097
       t2-F1-qe3-end-2014                         rates   1      1           1.943            1.45         1.062                1.002         0.9808
       t2-F1-sahm-watch-2024               unemployment   1      1               1               1             1                    1              1
       t2-F1-third-cut-2019                       rates   1      1           1.287           1.114         1.003                1.002         0.9671
       t2-F1-ust-positioning-2018                 rates   1      1           1.717           1.329         1.059                1.002          1.018
T2-F2  t2-F2-abenomics-2012                          fx   1 0.9656           1.352           1.203        0.9162               0.9626         0.9433
       t2-F2-act-as-appropriate-2019              rates   1      1            1.06          0.9825        0.9759               0.9882         0.9621
       t2-F2-aud-taper

## 7 — Save

In [8]:
M12.to_parquet(DATA / "m1_2_window_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_2_vs_m0_per_unit.csv")
FAMILY.to_csv(DATA / "m1_2_vs_m0_per_family.csv"); BY_TYPE.to_csv(DATA / "m1_2_vs_m0_by_asset_type.csv")
for f in ("m1_2_window_backtest_scores.parquet", "m1_2_vs_m0_per_unit.csv", "m1_2_vs_m0_per_family.csv", "m1_2_vs_m0_by_asset_type.csv"):
    print(f"data/{f:<36} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/m1_2_window_backtest_scores.parquet    4241.4 kB
data/m1_2_vs_m0_per_unit.csv                  15.0 kB
data/m1_2_vs_m0_per_family.csv                 0.7 kB
data/m1_2_vs_m0_by_asset_type.csv              0.9 kB


## 8 — In short

In [9]:
a = FAMILY.loc["all"]
for m in models:
    print(f"  {m:<26} {a[m]:.4f}   better than M0 on {better[m]}, better than M1.1 on {better_m11[m]}")
best = min((m for m in models if m.startswith("M1.2")), key=lambda m: a[m])
print(f"best M1.2 window on this block: {best} ({a[best]:.4f}); per family " +
      "; ".join(f"{f} {FAMILY.loc[f, best]:.4f}" for f in FAMILY.index if f != "all"))

  M0                         1.0000   better than M0 on 0/103, better than M1.1 on 20/103
  M1.1                       0.9976   better than M0 on 25/103, better than M1.1 on 0/103
  M1.2, 3 months             1.2775   better than M0 on 8/103, better than M1.1 on 6/103
  M1.2, 6 months             1.1081   better than M0 on 19/103, better than M1.1 on 19/103
  M1.2, 1 year               1.0065   better than M0 on 45/103, better than M1.1 on 40/103
  M1.2, 300 rows (M0)        0.9954   better than M0 on 62/103, better than M1.1 on 60/103
  M1.2, 2 years              0.9902   better than M0 on 64/103, better than M1.1 on 60/103
best M1.2 window on this block: M1.2, 2 years (0.9902); per family T2-F1 0.9700; T2-F2 0.9967; T2-F3 0.9761; T2-F4 1.0097
